In [ ]:
import glob
import os
import re
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pandas as pd
from scipy.stats import gaussian_kde
from scipy.signal import find_peaks
from mpl_toolkits.axes_grid1 import make_axes_locatable
import regionmask

In [ ]:
# Script is set up to run either TMAX or TMIN, but only one at a time. 

#OUTPUT_DIR = '...directory to TMAX files...'
#OUTPUT_DIR = '...directory to TMIN files...'
#OUTPUT_DIR = '/glade/derecho/scratch/cskinner/HeatSeasonLength/Baseline_Corrected/TMAX/'
OUTPUT_DIR = '/glade/derecho/scratch/cskinner/HeatSeasonLength/Baseline_Corrected/TMMN/'


def load_ensemble_data(period_tag):
    """
    Scans the output directory for processed files matching the period tag 
    ('Hist' or 'Fut'), loads them, and stacks them into a single Dataset.
    """
    search_pattern = os.path.join(OUTPUT_DIR, f"ExtremeHeat_*_{period_tag}*.nc")
    file_paths = sorted(glob.glob(search_pattern))
    
    if not file_paths:
        print(f"No files found for '{period_tag}'. Check your directory.")
        return None
        
    print(f"Found {len(file_paths)} files for the {period_tag} period. Stacking...")
    
    ds_list = []
    
    for file_path in file_paths:
        basename = os.path.basename(file_path)
        
        # Splitting 'ExtremeHeat_1011.001_Fut_2030_2059.nc' by '_' creates a list:
        # ['ExtremeHeat', '1011.001', 'Fut', '2030', '2059.nc']
        # Grab the second item (index 1)
        member_id = basename.split('_')[1] 
        
        ds = xr.open_dataset(file_path)
        ds = ds.expand_dims(member=[member_id])
        ds_list.append(ds)
        
    ds_ensemble = xr.concat(ds_list, dim='member')
    print(f"Successfully created ensemble dataset with dimensions: {dict(ds_ensemble.dims)}")
    
    return ds_ensemble


# ==========================================
# 1. LOAD THE DATA
# ==========================================
print("--- Loading Historical Ensemble ---")
ens_hist_raw = load_ensemble_data("Hist")

print("\n--- Loading Future Ensemble ---")
#ens_fut_raw = load_ensemble_data("Fut_1996_2025")
ens_fut_raw = load_ensemble_data("Fut_2026_2055")

# ==========================================
# 2. CALCULATE ENSEMBLE MEANS
# ==========================================

if ens_hist_raw is not None and ens_fut_raw is not None:
    print("\nCalculating Climatological Ensemble Means...")
    
    # Average across all years, then across all ensemble members
    ens_mean_hist = ens_hist_raw.mean(dim=['year', 'member'], skipna=True)
    ens_mean_fut  = ens_fut_raw.mean(dim=['year', 'member'], skipna=True)
    
    # Calculate the raw difference (Future - Historical)
    ens_mean_diff = ens_mean_fut - ens_mean_hist
    
    print("Done.")

In [ ]:
# This funciton was used to calculate the percent area of tropical land that exhibited at least a 330 day heat season in the future.

def calc_high_exposure_land_area(da_mean, threshold=330, lat_bounds=None):
    """
    Calculates the percentage of land area where the variable exceeds a given threshold,
    using cosine latitude weighting. If lat_bounds is provided as a tuple (min_lat, max_lat), 
    the calculation is restricted to that region.
    """
    # 1. Realign longitudes to -180 to 180 to match regionmask
    da_aligned = da_mean.assign_coords(lon=(((da_mean.lon + 180) % 360) - 180)).sortby('lon')
    
    # 2. Isolate the target latitude range
    if lat_bounds:
        da_aligned = da_aligned.where(
            (da_aligned.lat >= lat_bounds[0]) & (da_aligned.lat <= lat_bounds[1]), 
            drop=True
        )
        region_label = f"Latitudes {lat_bounds[0]} to {lat_bounds[1]}"
    else:
        region_label = "Global"

    print(f"\n--- Calculating {region_label} Land Area >= {threshold} Days ---")
    
    # 3. Apply the land mask (0 is land in natural_earth_v5_0_0)
    land_mask = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(da_aligned)
    da_land = da_aligned.where(land_mask == 0)
    
    # 4. Identify grid cells meeting the threshold
    exceeds_thresh = (da_land >= threshold)
    
    # 5. Calculate area weights (cosine of latitude)
    weights = np.cos(np.deg2rad(da_aligned.lat))
    
    # 6. Sum the weights for all valid land cells vs. cells exceeding the threshold
    total_land_weight = weights.where(da_land.notnull()).sum()
    exceeds_thresh_weight = weights.where(exceeds_thresh & da_land.notnull()).sum()
    
    pct_area = (exceeds_thresh_weight / total_land_weight) * 100
    
    print(f"{region_label} land area experiencing >= {threshold} days: {pct_area.values:.2f}%")
    return pct_area.values

# ==========================================
# EXECUTION 
# ==========================================
if ens_hist_raw is not None and ens_fut_raw is not None:
    # Average across all years, then across all ensemble members
    ens_mean_fut = ens_fut_raw.mean(dim=['year', 'member'], skipna=True)
    
    # Global land calculation
    pct_330_global = calc_high_exposure_land_area(
        ens_mean_fut['season_length'], 
        threshold=300
    )
    
    # Tropical land calculation (23.5°S to 23.5°N)
    pct_330_tropics = calc_high_exposure_land_area(
        ens_mean_fut['season_length'], 
        threshold=300, 
        lat_bounds=(-23.5, 23.5)
    )

In [ ]:
import regionmask
from mpl_toolkits.axes_grid1 import make_axes_locatable

def plot_average_season_length_with_spread(data_da, title="Average Extreme Heat Season Length", save_name="plots/CESM_SeasonLength.pdf"):
    """
    Plots the ensemble mean season length, matching the ERA5 PlateCarree formatting,
    with an aligned zonal mean plot on the right that includes the ensemble spread.
    
    IMPORTANT: data_da must retain the 'member' dimension (e.g., ens_hist_raw['season_length']).
    """
    print("1. Calculating temporal means...")
    
    # Average over time if the 'year' dimension exists
    if 'year' in data_da.dims:
        mem_mean = data_da.mean(dim='year', skipna=True)
    else:
        mem_mean = data_da

    print("2. Realigning Longitudes (0-360 to -180-180)...")
    mem_mean = mem_mean.assign_coords(lon=(((mem_mean.lon + 180) % 360) - 180)).sortby('lon')
    
    # Calculate the ultimate ensemble mean for the map plot
    ens_mean_map = mem_mean.mean(dim='member', skipna=True)
    
    print("3. Applying Land Mask for Zonal Calculation...")
    land_mask = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(mem_mean)
    
    # Mask both the individual members (for the zonal spread) and the ensemble mean (for the map)
    mem_mean_land = mem_mean.where(land_mask == 0)
    ens_mean_map_land = ens_mean_map.where(land_mask == 0)
    
    print("4. Calculating Zonal Mean and Full Ensemble Spread...")
    # Calculate zonal mean across longitudes for each individual member
    zonal_mem = mem_mean_land.mean(dim='lon', skipna=True)
    
    # Calculate the ensemble mean, min, and max of those zonal means
    zonal_ens_mean = zonal_mem.mean(dim='member', skipna=True)
    zonal_ens_min = zonal_mem.min(dim='member', skipna=True)
    zonal_ens_max = zonal_mem.max(dim='member', skipna=True)
    lats = zonal_ens_mean.lat

    # -------------------------------------------------------------------------
    # PLOTTING
    # -------------------------------------------------------------------------
    print(f"5. Generating perfectly aligned plot: {title}...")
    
    fig, ax_map = plt.subplots(figsize=(12, 7), subplot_kw={'projection': ccrs.PlateCarree()})
    ax_map.set_extent([-180, 180, -90, 90], crs=ccrs.PlateCarree())
    
    # Alignment
    divider = make_axes_locatable(ax_map)
    ax_zonal = divider.append_axes("right", size="20%", pad=0.3, axes_class=plt.Axes)
    cax = divider.append_axes("bottom", size="5%", pad=0.4, axes_class=plt.Axes)

    # ==========================================
    # LEFT: MASKED MAP PLOT
    # ==========================================
    map_plot = ens_mean_map_land.plot(
        ax=ax_map, transform=ccrs.PlateCarree(),
        cmap='YlOrRd',
        #levels=np.arange(0, 180, 20),
        #levels=np.arange(0, 320, 20),
        levels=np.arange(0, 360, 20),
        #levels=np.arange(0, 260, 20),
        add_colorbar=False,  
        extend='max',
        zorder=1
    )

    # Mask Ocean & Coastlines
    ax_map.add_feature(cfeature.OCEAN, facecolor='white', edgecolor='none', zorder=2)
    ax_map.coastlines(linewidth=0.8, color='black', zorder=3)
    ax_map.add_feature(cfeature.COASTLINE, linestyle=':', alpha=0.5, zorder=3)
    
    # Gridlines
    gl = ax_map.gridlines(
        draw_labels=True, 
        xlocs=np.arange(-180, 181, 60), 
        ylocs=np.arange(-90, 91, 30), 
        color='lightgray', 
        linewidth=0.8, 
        linestyle='-', 
        zorder=4
    )
    gl.top_labels = False
    gl.right_labels = False
    
    ax_map.set_title(title, fontsize=14, fontweight='bold', pad=15)
    
    # Colorbar
    cbar = plt.colorbar(map_plot, cax=cax, orientation='horizontal')
    cbar.set_label('Days', fontsize=12)
    
    # ==========================================
    # RIGHT: ZONAL MEAN PLOT
    # ==========================================
    # 
    ax_zonal.fill_betweenx(
        lats, 
        zonal_ens_min, 
        zonal_ens_max, 
        color='gray', alpha=0.3, zorder=1
    )
      
    # Ensemble Mean Line
    ax_zonal.plot(zonal_ens_mean, lats, color='red', linewidth=2, zorder=3)
    
    # Formatting
    ax_zonal.set_ylim(-90, 90) 
    #ax_zonal.set_xlim(0, 200)
    ax_zonal.set_xlim(0, 370)
    #ax_zonal.set_xlim(0, 300)
    ax_zonal.set_xticks(np.arange(0, 400, 100))
   # ax_zonal.set_xticks(np.arange(0, 350, 50))
    ax_zonal.set_yticks(np.arange(-90, 100, 30))
    ax_zonal.set_yticklabels(['90°S', '60°S', '30°S', 'EQ', '30°N', '60°N', '90°N'])
    ax_zonal.yaxis.tick_right()
    
    ax_zonal.set_xlabel('Days')
    ax_zonal.set_title('Land-Only Zonal Mean', fontsize=12, fontweight='bold')
    ax_zonal.grid(True, linestyle=':', alpha=0.6)
    
    if save_name:
        plt.savefig(save_name, format="pdf", bbox_inches="tight")
        
    plt.show()

In [ ]:
# ==========================================
# EXECUTION
# ==========================================
#plot_average_season_length_with_spread(
#    data_da=ens_fut_raw['season_length'], 
    #title="CESM Average TMAX (1996-2025) Extreme Heat Season Length",
    #title="CESM Average TMAX (2026-2055) Extreme Heat Season Length",
    #save_name="plots/CESM_TMAX_SeasonLength_1996-2025_withSpread.pdf"
    #save_name="plots/CESM_TMAX_SeasonLength_2026-2055_withSpread.pdf"
#)


# ==========================================
# EXECUTION
# ==========================================
plot_average_season_length_with_spread(
    data_da=ens_fut_raw['season_length'], 
    #title="CESM Average TMIN (1996-2025) Extreme Heat Season Length",
    title="CESM Average TMIN (2026-2055) Extreme Heat Season Length",
    #save_name="plots/CESM_TMIN_SeasonLength_1996-2025_withSpread.pdf"
    #save_name="plots/CESM_TMIN_SeasonLength_2026-2055_withSpread.pdf"
)

In [ ]:
def apply_fdr(p_values, alpha=0.05):
    """
    Applies the Benjamini-Hochberg False Discovery Rate (FDR) procedure.
    """
    p_flat = p_values.values.flatten()
    valid_idx = ~np.isnan(p_flat)
    p_valid = p_flat[valid_idx]
    
    # Sort p-values
    sorted_idx = np.argsort(p_valid)
    p_sorted = p_valid[sorted_idx]
    
    # Calculate critical values
    m = len(p_sorted)
    q_values = (np.arange(1, m + 1) / m) * alpha
    
    # Find the largest p-value that is less than its critical value
    significant = p_sorted <= q_values
    if np.any(significant):
        max_idx = np.where(significant)[0][-1]
        p_threshold = p_sorted[max_idx]
    else:
        p_threshold = -1.0 # No significant pixels
        
    # Create the spatial mask
    return p_values <= p_threshold


def calculate_emergence_robustness(da_hist, da_fut, land_mask, agreement_threshold=0.666, n_permutations=1000):
    """
    Calculates significance on a member-by-member basis (shuffling years).
    A grid cell is robust if >= threshold (e.g., 66%) of members show a 
    statistically significant change in the SAME direction as the ensemble mean.
    """
    print("1. Calculating Ensemble Mean Change & Direction...")
    # Calculate the mean across years for each member
    mem_mean_hist = da_hist.mean(dim='year', skipna=True)
    mem_mean_fut  = da_fut.mean(dim='year', skipna=True)
    
    # Calculate difference per member, then the overall ensemble mean
    mem_diff = mem_mean_fut - mem_mean_hist
    ens_mean_diff = mem_diff.mean(dim='member', skipna=True)
    
    # Get the overall direction of the ensemble mean change
    ens_sign = np.sign(ens_mean_diff.values)
    
    total_members = da_hist.sizes['member']
    n_years_hist = da_hist.sizes['year']
    n_years_total = n_years_hist + da_fut.sizes['year']
    
    # Initialize a blank map to count how many members pass the test
    robust_member_count = xr.zeros_like(ens_mean_diff)
    
    print(f"2. Running Member-by-Member Permutation Tests ({total_members} members, {n_permutations} shuffles each)...")
    np.random.seed(42)
    
    for m in range(total_members):
        print(f"   Processing Member {m+1}/{total_members}...")
        
        # Extract data for just this single member as pure numpy arrays for speed
        m_hist = da_hist.isel(member=m).values
        m_fut  = da_fut.isel(member=m).values
        m_obs_diff = mem_diff.isel(member=m).values
        
        # Get the direction of THIS member's change
        m_sign = np.sign(m_obs_diff)
        
        # Combine the 60 years for shuffling (axis=0 is the year dimension)
        combined = np.concatenate([m_hist, m_fut], axis=0) 
        exceed_count = np.zeros_like(m_obs_diff)
        
        # Shuffle years for this specific member
        for _ in range(n_permutations):
            idx = np.random.permutation(n_years_total)
            shuffled = combined[idx, ...]
            
            pseudo_hist = np.nanmean(shuffled[:n_years_hist, ...], axis=0)
            pseudo_fut  = np.nanmean(shuffled[n_years_hist:, ...], axis=0)
            pseudo_diff = pseudo_fut - pseudo_hist
            
            exceed_count += (np.abs(pseudo_diff) >= np.abs(m_obs_diff))
            
        # Calculate p-values
        p_values = exceed_count / n_permutations
        p_val_da = xr.DataArray(p_values, coords=ens_mean_diff.coords, dims=ens_mean_diff.dims)
        
        # Mask out oceans before running FDR
        p_val_da_land = p_val_da.where(land_mask == 0)
        m_fdr_mask = apply_fdr(p_val_da_land, alpha=0.05)
        # --------------------------------------------------------
        
        # Check criteria: Is it significant AND does the sign match the ensemble average?
        sign_match = (m_sign == ens_sign)
        member_is_robust = m_fdr_mask & sign_match
        
        # Add the passing grid cells to running tally
        robust_member_count += member_is_robust
        
    print(f"3. Applying {agreement_threshold*100:.1f}% Agreement Threshold...")
    final_emergence_mask = (robust_member_count / total_members) >= agreement_threshold
    
    return ens_mean_diff, final_emergence_mask



In [ ]:
print("\n--- PREPARING DATA & MASKING ---")
# 1. Shift longitudes upfront so regionmask works and drops duplicate seams
def shift_lon_and_clean(da):
    da_shifted = da.assign_coords(lon=(((da.lon + 180) % 360) - 180)).sortby('lon')
    return da_shifted.drop_duplicates(dim='lon')

da_len_hist = shift_lon_and_clean(ens_hist_raw['season_length'])
da_len_fut  = shift_lon_and_clean(ens_fut_raw['season_length'])

# 2. Generate the Land Mask ONCE
print("Generating land mask...")
land_mask = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(da_len_hist)

# 3. Run the member-by-member emergence
ens_diff, final_mask = calculate_emergence_robustness(
    da_len_hist, 
    da_len_fut, 
    land_mask=land_mask,       # <--- Pass the mask here
    agreement_threshold=0.666, 
    n_permutations=1000
)

In [ ]:
# NOW PLOT THE CHANGE

In [ ]:
import regionmask
from mpl_toolkits.axes_grid1 import make_axes_locatable

def plot_robust_map_with_zonal(da_hist, da_fut, ens_diff, robust_mask, title, 
                               label='Days', cmap='RdBu_r'):
    
    print("1. Realigning Longitudes (0-360 to -180-180)...")
    # Fix the Cartopy seam and prepare the data for regionmask
    def shift_lon(da):
        return da.assign_coords(lon=(((da.lon + 180) % 360) - 180)).sortby('lon')
        
    da_hist = shift_lon(da_hist)
    da_fut = shift_lon(da_fut)
    ens_diff = shift_lon(ens_diff)
    robust_mask = shift_lon(robust_mask)

    print("2. Applying Land Mask...")
    land_mask = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(da_hist)
    
    da_hist_land = da_hist.where(land_mask == 0)
    da_fut_land = da_fut.where(land_mask == 0)
    ens_diff_land = ens_diff.where(land_mask == 0)
    
    print("3. Calculating Zonal Means and Full Ensemble Spread (Land Only)...")
    mem_mean_hist = da_hist_land.mean(dim='year', skipna=True)
    mem_mean_fut  = da_fut_land.mean(dim='year', skipna=True)
    
    mem_diff = mem_mean_fut - mem_mean_hist
    zonal_mem_diff = mem_diff.mean(dim='lon', skipna=True)
    
    zonal_ens_mean = zonal_mem_diff.mean(dim='member', skipna=True)
    zonal_ens_min = zonal_mem_diff.min(dim='member', skipna=True)
    zonal_ens_max = zonal_mem_diff.max(dim='member', skipna=True)
    lats = zonal_ens_mean.lat

    print("4. Generating Aligned Figure...")
    fig, ax_map = plt.subplots(figsize=(12, 7), subplot_kw={'projection': ccrs.PlateCarree()})
    ax_map.set_extent([-180, 180, -90, 90], crs=ccrs.PlateCarree())
    
    # Create dividers for alignment
    divider = make_axes_locatable(ax_map)
    ax_zonal = divider.append_axes("right", size="20%", pad=0.3, axes_class=plt.Axes)
    cax = divider.append_axes("bottom", size="5%", pad=0.4, axes_class=plt.Axes)

    # ==========================================
    # LEFT: MASKED MAP PLOT
    # ==========================================
    robust_diff = ens_diff_land.where(robust_mask)
    
    # CUSTOM BINS
    #custom_bins = [-260, -220, -180, -160, -140, -120, -100, -80, -60, -50, -40, -30, -20, -10, 
                #   0, 10, 20, 30, 40, 50, 60, 80, 100, 120, 140, 160, 180, 220, 260]

   # custom_bins = [-180, -160, -140, -120, -100, -80, -60, -50, -40, -30, -20, -10, 0, 10, 20, 30, 40, 50, 60, 80, 100, 120, 140, 160, 180]
    
    map_plot = robust_diff.plot(
        ax=ax_map, 
        transform=ccrs.PlateCarree(),
        cmap=cmap, 
        #levels=np.arange(-150, 165, 15), 
        levels=np.arange(-180, 200, 20), 
        #levels=np.arange(-120, 130, 10), 
        extend='both',
        add_colorbar=False, # We handle this manually below
        zorder=1
    )
    
    # MAP FORMATTING
    ax_map.add_feature(cfeature.OCEAN, facecolor='white', edgecolor='none', zorder=2)
    ax_map.coastlines(color='black', linewidth=0.8, zorder=3)
    ax_map.add_feature(cfeature.COASTLINE, linestyle=':', edgecolor='gray', linewidth=0.5, zorder=3)
    
    # Gridlines
    gl = ax_map.gridlines(
        draw_labels=True, 
        xlocs=np.arange(-180, 181, 60), 
        ylocs=np.arange(-90, 91, 30), 
        color='lightgray', 
        linewidth=0.8, 
        linestyle='-', 
        zorder=4
    )
    gl.top_labels = False
    gl.right_labels = False
    
    ax_map.set_title(title, fontsize=14, fontweight='bold', pad=15)
    
    # Colorbar at the bottom
    #cbar = plt.colorbar(map_plot, cax=cax, orientation='horizontal', ticks=np.arange(-150, 151, 30))
    cbar = plt.colorbar(map_plot, cax=cax, orientation='horizontal', ticks=np.arange(-180, 181, 60))
    #cbar = plt.colorbar(map_plot, cax=cax, orientation='horizontal', ticks=np.arange(-120, 121, 40))
    cbar.set_label(label, fontsize=12)

    
    # ==========================================
    # RIGHT: ZONAL MEAN PLOT
    # ==========================================
    ax_zonal.axvline(0, color='black', linestyle='-', linewidth=1, zorder=2)
    
    # Full Ensemble Spread
    ax_zonal.fill_betweenx(
        lats, 
        zonal_ens_min, 
        zonal_ens_max, 
        color='gray', alpha=0.3, zorder=1    )
      
    # Ensemble Mean Line
    ax_zonal.plot(zonal_ens_mean, lats, color='red', linewidth=2, zorder=3)
    
    # Formatting
    ax_zonal.set_ylim(-90, 90) 
    ax_zonal.set_xlim(0, 260)   
    #ax_zonal.set_xlim(-20, 150)  
    ax_zonal.set_yticks(np.arange(-90, 100, 30))
    ax_zonal.set_yticklabels(['90°S', '60°S', '30°S', 'EQ', '30°N', '60°N', '90°N'])
    ax_zonal.yaxis.tick_right()
    
    ax_zonal.set_xlabel(label)
    ax_zonal.set_title('Land-Only Zonal Mean', fontsize=12, fontweight='bold')
    ax_zonal.grid(True, linestyle=':', alpha=0.6)
    
    # Optional: Save the plot
    #plt.savefig("plots/CESM_TMIN_Extreme_Heat_Season_Length_Change_96_25-66_95_Zonal_Panel.pdf", format="pdf", bbox_inches="tight")
    #plt.savefig("plots/CESM_TMIN_Extreme_Heat_Season_Length_Change_26_55-66_95_Zonal_Panel.pdf", format="pdf", bbox_inches="tight")
    #plt.savefig("plots/CESM_TMAX_Extreme_Heat_Season_Length_Change_96_25-66_95_Zonal_Panel.pdf", format="pdf", bbox_inches="tight")
    #plt.savefig("plots/CESM_TMAX_Extreme_Heat_Season_Length_Change_26_55-66_95_Zonal_Panel.pdf", format="pdf", bbox_inches="tight")

    
    plt.show()
    return fig

# Execute the plot
fig = plot_robust_map_with_zonal(
    da_hist=da_len_hist,
    da_fut=da_len_fut,
    ens_diff=ens_diff, 
    robust_mask=final_mask,
    #title="Avg CESM Change in TMAX Extreme Heat Season Length (1996-2025) - (1966-1995)",
    #title="Avg CESM Change in TMAX Extreme Heat Season Length (2026-2055) - (1966-1995)",
    #title="Avg CESM Change in TMIN Extreme Heat Season Length (1996-2025) - (1966-1995)",
    title="Avg CESM Change in TMIN Extreme Heat Season Length (2026-2055) - (1966-1995)",
    label="Days",
    cmap="RdBu_r"
)

In [ ]:
def plot_robust_percent_change_with_zonal(da_hist, da_fut, robust_mask, title, 
                                          label='Percent Change (%)', cmap='RdBu_r', 
                                          save_name=None):
    
    print("1. Realigning Longitudes (0-360 to -180-180)...")
    # Fix the Cartopy seam and prepare the data for regionmask
    def shift_lon(da):
        return da.assign_coords(lon=(((da.lon + 180) % 360) - 180)).sortby('lon')
        
    da_hist = shift_lon(da_hist)
    da_fut = shift_lon(da_fut)
    robust_mask = shift_lon(robust_mask)

    print("2. Applying Land Mask...")
    land_mask = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(da_hist)
    
    da_hist_land = da_hist.where(land_mask == 0)
    da_fut_land = da_fut.where(land_mask == 0)
    
    print("3. Calculating Percent Change and Ensemble Spread (Land Only)...")
    mem_mean_hist = da_hist_land.mean(dim='year', skipna=True)
    mem_mean_fut  = da_fut_land.mean(dim='year', skipna=True)
    
    # --- THE PERCENT MATH ---
    # We use xr.where to ensure we only divide where historical data > 0
    # to avoid Infinity errors where the historical season length was 0 days.
    mem_pct_change = xr.where(
        mem_mean_hist > 0, 
        ((mem_mean_fut - mem_mean_hist) / mem_mean_hist) * 100.0, 
        np.nan
    )
    
    # Calculate the ensemble mean for the Map
    ens_mean_pct = mem_pct_change.mean(dim='member', skipna=True)
    
    # Calculate the Zonal Spread for the line graph
    zonal_mem_pct = mem_pct_change.mean(dim='lon', skipna=True)
    zonal_ens_mean = zonal_mem_pct.mean(dim='member', skipna=True)
    zonal_ens_min = zonal_mem_pct.min(dim='member', skipna=True)
    zonal_ens_max = zonal_mem_pct.max(dim='member', skipna=True)
    lats = zonal_ens_mean.lat

    print("4. Generating Aligned Percentage Figure...")
    fig, ax_map = plt.subplots(figsize=(12, 7), subplot_kw={'projection': ccrs.PlateCarree()})
    ax_map.set_extent([-180, 180, -90, 90], crs=ccrs.PlateCarree())
    
    divider = make_axes_locatable(ax_map)
    ax_zonal = divider.append_axes("right", size="20%", pad=0.3, axes_class=plt.Axes)
    cax = divider.append_axes("bottom", size="5%", pad=0.4, axes_class=plt.Axes)

    # ==========================================
    # LEFT: MASKED PERCENTAGE MAP
    # ==========================================
    # Apply the absolute robustness mask
    robust_pct = ens_mean_pct.where(robust_mask)
    
    map_plot = robust_pct.plot(
        ax=ax_map, 
        transform=ccrs.PlateCarree(),
        cmap=cmap, 
        levels=np.arange(-250, 275, 25),
        #levels=np.arange(-150, 165, 15),
        extend='both',
        add_colorbar=False, 
        zorder=1
    )
    
    # MAP FORMATTING
    ax_map.add_feature(cfeature.OCEAN, facecolor='white', edgecolor='none', zorder=2)
    ax_map.coastlines(color='black', linewidth=0.8, zorder=3)
    ax_map.add_feature(cfeature.COASTLINE, linestyle=':', edgecolor='gray', linewidth=0.5, zorder=3)
    
    # Gridlines
    gl = ax_map.gridlines(
        draw_labels=True, 
        xlocs=np.arange(-180, 181, 60), 
        ylocs=np.arange(-90, 91, 30), 
        color='lightgray', 
        linewidth=0.8, 
        linestyle='-', 
        zorder=4
    )
    gl.top_labels = False
    gl.right_labels = False
    
    ax_map.set_title(title, fontsize=14, fontweight='bold', pad=15)

    # ==========================================
    # COLORBAR
    # ==========================================
    # Define the exact tick intervals you want to display
    #custom_ticks = np.arange(-150, 151, 30)
    custom_ticks = np.arange(-250, 251, 50)
    
    cbar = plt.colorbar(map_plot, cax=cax, orientation='horizontal', ticks=custom_ticks)
    cbar.set_label(label, fontsize=12)
    
    #cbar = plt.colorbar(map_plot, cax=cax, orientation='horizontal')
    #cbar.set_label(label, fontsize=12)
    
    # ==========================================
    # RIGHT: ZONAL PERCENTAGE PLOT
    # ==========================================
    ax_zonal.axvline(0, color='black', linestyle='-', linewidth=1, zorder=2)
    
    ax_zonal.fill_betweenx(
        lats, 
        zonal_ens_min, 
        zonal_ens_max, 
        color='gray', alpha=0.3, zorder=1
    )
     
    ax_zonal.plot(zonal_ens_mean, lats, color='red', linewidth=2, zorder=3)
    
    ax_zonal.set_ylim(-90, 90) 
    ax_zonal.set_xlim(-20, 200)   
    ax_zonal.set_xlim(0, 450)
    #custom_ticks =[0, 50, 100, 150, 200]
    #ax_zonal.set_xticks(custom_ticks)
    ax_zonal.set_xticks(np.arange(0, 500, 100))
    ax_zonal.set_yticks(np.arange(-90, 100, 30))
    ax_zonal.set_yticklabels(['90°S', '60°S', '30°S', 'EQ', '30°N', '60°N', '90°N'])
    ax_zonal.yaxis.tick_right()
    
    ax_zonal.set_xlabel(label)
    ax_zonal.set_title('Land-Only Zonal Mean (%)', fontsize=12, fontweight='bold')
    ax_zonal.grid(True, linestyle=':', alpha=0.6)
    
    if save_name:
        plt.savefig(save_name, format="pdf", bbox_inches="tight")
    
    plt.show()
    return fig

# Execute the plot
fig_pct = plot_robust_percent_change_with_zonal(
    da_hist=da_len_hist, 
    da_fut=da_len_fut, 
    robust_mask=final_mask, 
    #title="Avg CESM Percent Change in TMAX Extreme Heat Season Length (2026-2055) - (1966-1995)",
    #title="Avg CESM Percent Change in TMAX Extreme Heat Season Length (1996-2025) - (1966-1995)",
    title="Avg CESM Percent Change in TMIN Extreme Heat Season Length (2026-2055) - (1966-1995)",
    #title="Avg CESM Percent Change in TMIN Extreme Heat Season Length (1996-2025) - (1966-1995)",
    label="Percent Change (%)",
    cmap="RdBu_r",
    #save_name="plots/CESM_TMAX_Percent_Change_Season_Length_26_55-66_95_Zonal_Panel.pdf"
    #save_name="plots/CESM_TMAX_Percent_Change_Season_Length_96_25-66_95_Zonal_Panel.pdf"
    #save_name="plots/CESM_TMIN_Percent_Change_Season_Length_26_55-66_95_Zonal_Panel.pdf"
    #save_name="plots/CESM_TMIN_Percent_Change_Season_Length_96_25-66_95_Zonal_Panel.pdf"
)

In [ ]:
# This code was used to calcuate fractions of land area for a given change presented in the manuscript text

# =========================================================================================
#  REGIONAL AVERAGES & FRACTIONAL STATISTICS
# =========================================================================================
print("\n" + "="*60)
print(" CALCULATING UNMASKED REGIONAL AVERAGES")
print("="*60)

# 1. Isolate land-only pixels for absolute shift
diff_land = ens_diff.where(land_mask == 0)

# 2. Calculate Percent Change (Ensemble Mean Future - Ensemble Mean Hist)
mean_hist = da_len_hist.mean(dim=['year', 'member'], skipna=True)
pct_change = xr.where(mean_hist > 0, (ens_diff / mean_hist) * 100.0, np.nan)
pct_land = pct_change.where(land_mask == 0)

# 3. Create area weights (cosine of latitude)
weights = np.cos(np.deg2rad(diff_land.lat))

def get_weighted_mean(da_region):
    return da_region.weighted(weights).mean(dim=['lat', 'lon'], skipna=True).compute().item()

def print_regional_stats(data_map, unit):
    global_da   = data_map
    tropical_da = data_map.where(abs(data_map.lat) <= 23.5)
    midlat_da   = data_map.where((abs(data_map.lat) > 23.5) & (abs(data_map.lat) <= 66.5))
    highlat_da  = data_map.where(abs(data_map.lat) > 66.5)
    
    print(f" Global Land Average:        {get_weighted_mean(global_da):>7.2f} {unit}")
    print(f" Tropical Land Average:      {get_weighted_mean(tropical_da):>7.2f} {unit} (|lat| <= 23.5)")
    print(f" Midlatitude Land Average:   {get_weighted_mean(midlat_da):>7.2f} {unit} (23.5 < |lat| <= 66.5)")
    print(f" High Latitude Land Average: {get_weighted_mean(highlat_da):>7.2f} {unit} (|lat| > 66.5)")

print("\n--- ABSOLUTE CHANGE ---")
print_regional_stats(diff_land, "days")

print("\n--- PERCENT CHANGE ---")
print_regional_stats(pct_land, "%")


print("\n" + "="*60)
print(" SIGNIFICANT LAND AREA & PIXEL FRACTIONS")
print("="*60)

is_land = (land_mask == 0) & ens_diff.notnull()
land_weights = weights * xr.ones_like(ens_diff).where(is_land)

total_land_area = land_weights.sum().compute().item()
total_land_pixels = is_land.sum().compute().item()

sig_increase = final_mask & (ens_diff > 0) & is_land
sig_decrease = final_mask & (ens_diff < 0) & is_land

# 1. Area Fractions
area_increase = land_weights.where(sig_increase).sum().compute().item()
area_decrease = land_weights.where(sig_decrease).sum().compute().item()
pct_area_inc = (area_increase / total_land_area) * 100
pct_area_dec = (area_decrease / total_land_area) * 100
pct_area_none = 100 - (pct_area_inc + pct_area_dec)

# 2. Pixel Fractions
pix_increase = sig_increase.sum().compute().item()
pix_decrease = sig_decrease.sum().compute().item()
pct_pix_inc = (pix_increase / total_land_pixels) * 100
pct_pix_dec = (pix_decrease / total_land_pixels) * 100
pct_pix_none = 100 - (pct_pix_inc + pct_pix_dec)

print("\n--- PHYSICAL LAND AREA FRACTIONS ---")
print(f" Sig. Increase (Longer):   {pct_area_inc:>5.1f}% of global land area")
print(f" Sig. Decrease (Shorter):  {pct_area_dec:>5.1f}% of global land area")
print(f" No Significant Change:    {pct_area_none:>5.1f}% of global land area")

print("\n--- LAND GRID CELL (PIXEL) FRACTIONS ---")
print(f" Sig. Increase (Longer):   {pct_pix_inc:>5.1f}% of land grid cells")
print(f" Sig. Decrease (Shorter):  {pct_pix_dec:>5.1f}% of land grid cells")
print(f" No Significant Change:    {pct_pix_none:>5.1f}% of land grid cells")
print("="*60 + "\n")

In [ ]:
# This code was used to examine the maximum projection of season length change among the ensemble members. It then compares that maximum
# value to the ensemble mean value.


import regionmask
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.axes_grid1 import make_axes_locatable


print("\n--- Calculating Maximum Ensemble Change ---")

# 1. Realign longitudes (0-360 to -180-180) so the land mask applies correctly
def shift_lon_and_clean(da):
    da_shifted = da.assign_coords(lon=(((da.lon + 180) % 360) - 180)).sortby('lon')
    return da_shifted.drop_duplicates(dim='lon')

da_len_hist = shift_lon_and_clean(ens_hist_raw['season_length'])
da_len_fut  = shift_lon_and_clean(ens_fut_raw['season_length'])

# 2. Get the per-member climatologies (average across years)
mem_mean_hist = da_len_hist.mean(dim='year', skipna=True)
mem_mean_fut  = da_len_fut.mean(dim='year', skipna=True)

# 3. Calculate the raw change for each individual member
mem_diff = mem_mean_fut - mem_mean_hist

# 4. Find the maximum absolute change across the ensemble for each grid cell
max_diff = mem_diff.max(dim='member', skipna=True)

# 5. Identify the index of the member that produced the maximum change
max_idx = mem_diff.fillna(-9999).argmax(dim='member')

# 6. Extract the historical baseline specifically for the member that had the max change
hist_at_max_change = mem_mean_hist.isel(member=max_idx)

# 7. Calculate the percentage change associated with that specific maximum jump
max_associated_pct = xr.where(
    hist_at_max_change > 0, 
    (max_diff / hist_at_max_change) * 100.0, 
    np.nan
)

# 8. Generate the land mask using the shifted coordinates
land_mask = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(da_len_hist)

# 9. Apply land mask to the final variables
max_diff_land = max_diff.where(land_mask == 0)
max_associated_pct_land = max_associated_pct.where(land_mask == 0)


print("\n--- Calculating Ensemble Means & Differences ---")

# 1. Calculate Ensemble Mean Change (Days)
ens_mean_diff = mem_diff.mean(dim='member', skipna=True)

# Difference: Max Days - Mean Days
diff_from_mean_days = max_diff - ens_mean_diff

# 2. Calculate Ensemble Mean Percentage Change 
mem_pct_change = xr.where(
    mem_mean_hist > 0, 
    ((mem_mean_fut - mem_mean_hist) / mem_mean_hist) * 100.0, 
    np.nan
)
ens_mean_pct = mem_pct_change.mean(dim='member', skipna=True)

# Difference: Max Pct - Mean Pct
diff_from_mean_pct = max_associated_pct - ens_mean_pct

# 3. Apply the Land Mask to the new difference variables
diff_from_mean_days_land = diff_from_mean_days.where(land_mask == 0)
diff_from_mean_pct_land = diff_from_mean_pct.where(land_mask == 0)





def plot_max_and_diff_maps(data_left, data_right, main_title,
                           title_left="Maximum Change", title_right="Difference (Max - Mean)",
                           label_left='Days', label_right='Days',
                           cmap_left='RdBu_r', cmap_right='YlOrRd',
                           levels_left=np.arange(-120, 130, 10), 
                           levels_right=np.arange(0, 65, 5),
                           cbar_ticks_left=np.arange(-120, 121, 40),
                           cbar_ticks_right=np.arange(0, 61, 15),
                           save_name=None):
    
    print(f"Generating Two-Panel Figure: {main_title}...")
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 7), subplot_kw={'projection': ccrs.PlateCarree()})
    fig.suptitle(main_title, fontsize=16, fontweight='bold', y=0.98)
    
    # ==========================================
    # LEFT PANEL: MAXIMUM CHANGE
    # ==========================================
    ax1 = axes[0]
    ax1.set_extent([-180, 180, -90, 90], crs=ccrs.PlateCarree())
    
    plot1 = data_left.plot(
        ax=ax1, transform=ccrs.PlateCarree(), cmap=cmap_left, 
        levels=levels_left, extend='both', add_colorbar=False, zorder=1
    )
    
    ax1.add_feature(cfeature.OCEAN, facecolor='white', edgecolor='none', zorder=2)
    ax1.coastlines(color='black', linewidth=0.8, zorder=3)
    ax1.add_feature(cfeature.COASTLINE, linestyle=':', edgecolor='gray', linewidth=0.5, zorder=3)
    
    gl1 = ax1.gridlines(draw_labels=True, xlocs=np.arange(-180, 181, 60), ylocs=np.arange(-90, 91, 30), color='lightgray', linewidth=0.8, linestyle='-', zorder=4)
    gl1.top_labels = False; gl1.right_labels = False
    ax1.set_title(title_left, fontsize=13, fontweight='bold', pad=10)
    
    # Colorbar 1
    divider1 = make_axes_locatable(ax1)
    cax1 = divider1.append_axes("bottom", size="5%", pad=0.4, axes_class=plt.Axes)
    cbar1 = plt.colorbar(plot1, cax=cax1, orientation='horizontal', ticks=cbar_ticks_left)
    cbar1.set_label(label_left, fontsize=12)

    # ==========================================
    # RIGHT PANEL: DIFFERENCE (MAX - MEAN)
    # ==========================================
    ax2 = axes[1]
    ax2.set_extent([-180, 180, -90, 90], crs=ccrs.PlateCarree())
    
    plot2 = data_right.plot(
        ax=ax2, transform=ccrs.PlateCarree(), cmap=cmap_right, 
        levels=levels_right, extend='max', add_colorbar=False, zorder=1
    )
    
    ax2.add_feature(cfeature.OCEAN, facecolor='white', edgecolor='none', zorder=2)
    ax2.coastlines(color='black', linewidth=0.8, zorder=3)
    ax2.add_feature(cfeature.COASTLINE, linestyle=':', edgecolor='gray', linewidth=0.5, zorder=3)
    
    gl2 = ax2.gridlines(draw_labels=True, xlocs=np.arange(-180, 181, 60), ylocs=np.arange(-90, 91, 30), color='lightgray', linewidth=0.8, linestyle='-', zorder=4)
    gl2.top_labels = False; gl2.right_labels = False
    gl2.left_labels = False  # Turn off left labels to keep the center space clean
    ax2.set_title(title_right, fontsize=13, fontweight='bold', pad=10)
    
    # Colorbar 2
    divider2 = make_axes_locatable(ax2)
    cax2 = divider2.append_axes("bottom", size="5%", pad=0.4, axes_class=plt.Axes)
    cbar2 = plt.colorbar(plot2, cax=cax2, orientation='horizontal', ticks=cbar_ticks_right)
    cbar2.set_label(label_right, fontsize=12)
    
    plt.subplots_adjust(wspace=0.05)
    
    if save_name:
        plt.savefig(save_name, format="pdf", bbox_inches="tight")
    
    plt.show()
    return fig


# 1. Plot the Absolute Change (Days)
fig_days = plot_max_and_diff_maps(
    data_left=max_diff_land,
    data_right=diff_from_mean_days_land,
    main_title="CESM TMIN Season Length: Max Change vs. Ensemble Mean",
    title_left="Maximum Ensemble Change",
    title_right="Difference (Max Member - Ensemble Mean)",
    label_left="Days",
    label_right="Days",
    cmap_left="RdBu_r",
    cmap_right="Reds",
    levels_left=np.arange(-180, 200, 20),
    levels_right=np.arange(0, 50, 5),        # 0 to 50 extra days
    cbar_ticks_left=np.arange(-180, 181, 60),
    cbar_ticks_right=np.arange(0, 51, 10),
    #save_name="plots/CESM_TMIN_EnsembleMaximumChange_and_Diff_from_EnsembleMean_Season_Length_Days_26_55-66_95.pdf"
)

# 2. Plot the Associated Percentage Change
fig_pct = plot_max_and_diff_maps(
    data_left=max_associated_pct_land,
    data_right=diff_from_mean_pct_land,
    main_title="CESM TMIN Season Length Percentage: Max vs. Ensemble Mean",
    title_left="Associated Max Percent Change",
    title_right="Difference (Max % - Ensemble Mean %)",
    label_left="Percent Change (%)",
    label_right="Percentage Points",
    cmap_left="RdBu_r",
    cmap_right="Reds",
    levels_left=np.arange(-250, 275, 25),
    levels_right=np.arange(0, 110, 10),       # 0 to 100 extra percent
    cbar_ticks_left=np.arange(-250, 251, 50),
    cbar_ticks_right=np.arange(0, 101, 20),
    #save_name="plots/CESM_TMIN_EnsembleMaximumChange_and_Diff_from_EnsembleMean_Season_Length_Percentage_26_55-66_95.pdf"
)